# NB06 · Semana 14 — Serving: gold y features en PostgreSQL + tablero en Metabase

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 4 (E4)**

## Objetivo

Cerrar el pipeline: la gold y los features llegan a PostgreSQL y desde ahí los lee Metabase.

```
bronze (Delta, MinIO) ─▶ dbt: stg_urlhaus ─▶ fct_familias_mensual ─▶ fct_amenazas_mensual
                                                      └─▶ fct_features_familias
                                   │
            DAG pipeline_grupo07 · tarea publicar_gold (todos los días, una transacción)
                                   ▼
            PostgreSQL · warehouse.grupo07.*  ─▶  Metabase
```

**La publicación la hace el DAG, no este notebook.** Acá se verifica lo que quedó publicado, se retira la vista vieja y se deja la guía del tablero.

**¿Por qué mover la gold del lakehouse al warehouse?** DuckDB corre dentro de un proceso y lee archivos del lake: no atiende a varios usuarios a la vez ni maneja permisos. PostgreSQL sí: Metabase se conecta con el usuario del grupo, ve solo el esquema `grupo07` y lee tablas chicas, ya agregadas, que responden rápido.

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import PG, info, pg_url
info()

In [ ]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

## 1. Qué publicó el DAG

Las tres tablas tienen que existir, no estar vacías y **sumar la misma cantidad de URLs**: todas salen de la misma silver (lo verifica también el test `assert_gold_cuadra` antes de publicar).

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

ESQUEMA = os.environ.get("PG_SCHEMA") or PG["user"]          # grupo07
TABLAS = ["fct_familias_mensual", "fct_amenazas_mensual", "fct_features_familias"]
engine = create_engine(pg_url())

filas = []
with engine.connect() as c:
    for t in TABLAS:
        n, urls, desde, hasta, publicado = c.execute(text(
            f'select count(*), sum(cantidad_amenazas), min(make_date(anio, mes, 1)), '
            f'max(make_date(anio, mes, 1)), max(actualizado_en) from "{ESQUEMA}".{t}')).one()
        filas.append({"tabla": t, "filas": n, "urls": int(urls), "desde": desde, "hasta": hasta,
                      "publicado": publicado})
publicado = pd.DataFrame(filas)
print(publicado.to_string(index=False))
assert (publicado.filas > 0).all(), "Hay una tabla vacía"
assert publicado.urls.nunique() == 1, "Las tablas no suman lo mismo"
print(f"\n✅ Las tres tablas suman {publicado.urls.iloc[0]} URLs")

## 2. Por qué la publicación no duplica

`publicar_gold` hace `TRUNCATE` + `INSERT` de cada tabla **dentro de una sola transacción**: si algo falla a mitad de camino, PostgreSQL vuelve a la versión anterior completa. Correrla dos veces deja exactamente lo mismo (lo probamos publicando dos veces seguidas contra un PostgreSQL de prueba). Además cada tabla tiene **clave primaria** en su grano, así que una fila duplicada ni siquiera se puede insertar:

In [ ]:
with engine.connect() as c:
    claves = c.execute(text("""
        select tc.table_name, string_agg(kcu.column_name, ', ' order by kcu.ordinal_position)
        from information_schema.table_constraints tc
        join information_schema.key_column_usage kcu
          on kcu.constraint_name = tc.constraint_name and kcu.table_schema = tc.table_schema
        where tc.table_schema = :e and tc.constraint_type = 'PRIMARY KEY'
        group by 1 order by 1"""), {"e": ESQUEMA}).fetchall()
for tabla, cols in claves:
    print(f"{tabla:24} clave primaria: ({cols})")

## 3. Features

Están definidos **una sola vez**, en el modelo dbt `fct_features_familias` (lo escribe NB05, sección 3c). Grano: una fila por mes y familia de malware, con los meses sin actividad completados con 0.

| Feature | Qué es |
|---|---|
| `cantidad_mes_anterior` | URLs de la familia el mes calendario anterior |
| `variacion_mensual_pct` | cambio % contra el mes anterior (vacío si el anterior fue 0) |
| `promedio_movil_3m` | promedio del mes y los dos anteriores |
| `acumulado` | total desde la primera aparición de la familia |
| `pct_del_mes` | participación de la familia en las URLs del mes |
| `mes_completo` | falso para el último mes, que puede estar incompleto |

**Sin fuga temporal:** todas las ventanas miran solo hacia atrás (`lag`, `rows between 2 preceding and current row`, `unbounded preceding`). El feature de marzo usa marzo, febrero y enero, nunca abril. El test `assert_features_sin_fuga` los recalcula desde cero filtrando `fecha <= mes del feature` y falla si no coinciden; lo probamos cambiando la ventana a `1 following` y el test detectó la fuga.

**Reproducibles:** es SQL versionado que se recalcula entero desde la gold en cada corrida: con los mismos datos, da lo mismo.

Familias que más crecieron en el último mes completo (con al menos 20 URLs, para no confundir ruido con tendencia):

In [ ]:
with engine.connect() as c:
    crecen = pd.read_sql(text(f"""
        select familia_malware, fecha, cantidad_amenazas, cantidad_mes_anterior,
               variacion_mensual_pct, promedio_movil_3m, pct_del_mes
        from "{ESQUEMA}".fct_features_familias
        where fecha = (select max(fecha) from "{ESQUEMA}".fct_features_familias where mes_completo)
          and cantidad_amenazas >= 20
        order by variacion_mensual_pct desc nulls first
        limit 10"""), c)
crecen

## 4. Retirar la vista vieja

Antes los features eran una vista (`v_amenazas_features`) que creaba este notebook a mano, sin tests y calculada por tipo de amenaza (URLhaus publica un solo tipo, así que `pct_del_mes` daba siempre 100 %). La reemplaza `fct_features_familias`, que viene de dbt. Se borra para que no queden dos definiciones de los mismos features.

In [ ]:
with engine.begin() as c:
    c.execute(text(f'drop view if exists "{ESQUEMA}".v_amenazas_features'))
print("Vista vieja retirada (si existía).")

## 5. Documentación dentro del warehouse

`publicar_gold` copia las descripciones de `schema.yml` como `COMMENT` de PostgreSQL: Metabase las muestra al lado de cada columna, y la documentación se escribe una sola vez (en dbt).

In [ ]:
with engine.connect() as c:
    comentarios = pd.read_sql(text("""
        select c.relname as tabla, a.attname as columna, col_description(a.attrelid, a.attnum) as descripcion
        from pg_attribute a
        join pg_class c on c.oid = a.attrelid
        join pg_namespace n on n.oid = c.relnamespace
        where n.nspname = :e and c.relname = 'fct_features_familias' and a.attnum > 0 and not a.attisdropped
        order by a.attnum"""), c, params={"e": ESQUEMA})
pd.set_option("display.max_colwidth", 120)
comentarios

## 6. Tablero en Metabase

En **https://metabase.nexoacademia.org**, base `warehouse`, esquema `grupo07`. Como no somos admin, si las tablas nuevas no aparecen en *Explorar datos*, usar **Nuevo → Consulta SQL** con las consultas de abajo.

Dashboard: **"Amenazas URLhaus — grupo07"**. Cada gráfico responde una pregunta; el título del gráfico es la pregunta.

**1 · ¿Cuántas URLs maliciosas nuevas se registran por mes?** — línea
```sql
select make_date(anio, mes, 1) as mes, sum(cantidad_amenazas) as urls_nuevas
from grupo07.fct_amenazas_mensual
group by 1 order by 1
```

**2 · ¿Qué familias de malware fueron las más activas en los últimos 3 meses completos?** — barras horizontales
```sql
select familia_malware as familia, sum(cantidad_amenazas) as urls
from grupo07.fct_features_familias
where mes_completo and familia_malware <> 'sin_familia'
  and fecha > (select max(fecha) from grupo07.fct_features_familias where mes_completo) - interval '3 months'
group by 1 order by 2 desc limit 10
```

**3 · ¿Qué familias están creciendo?** — tabla (último mes completo, familias con al menos 20 URLs)
```sql
select familia_malware as familia, cantidad_amenazas as urls_del_mes,
       cantidad_mes_anterior as urls_mes_anterior, variacion_mensual_pct as variacion_pct,
       promedio_movil_3m as promedio_3_meses
from grupo07.fct_features_familias
where fecha = (select max(fecha) from grupo07.fct_features_familias where mes_completo)
  and cantidad_amenazas >= 20 and familia_malware <> 'sin_familia'
order by variacion_mensual_pct desc nulls first
```

**4 · ¿Con qué tipo de archivo se distribuye cada familia?** — barras apiladas (familia × tipo de archivo)
```sql
with top as (
    select familia_malware from grupo07.fct_familias_mensual
    where familia_malware <> 'sin_familia'
    group by 1 order by sum(cantidad_amenazas) desc limit 8
)
select f.familia_malware as familia, f.tipo_archivo, sum(f.cantidad_amenazas) as urls
from grupo07.fct_familias_mensual f join top using (familia_malware)
group by 1, 2 order by 1, 3 desc
```

**Qué decisión se toma con esto:** un equipo de seguridad prioriza reglas de bloqueo y firmas para las familias que más crecen (gráficos 2 y 3) y para los formatos por los que llegan (gráfico 4: si una familia se distribuye como `elf`, apunta a dispositivos Linux/IoT; si es `exe` o `zip`, a usuarios de Windows).

**Para leer bien los datos:** la fecha es cuándo URLhaus *registró* la URL, no cuándo ocurrió el ataque; el último mes está incompleto (por eso los gráficos 2 y 3 usan `mes_completo`).

Para la entrega: sacar una captura del dashboard y guardarla en `docs/tablero.png` del repo.